# Generate candidate scenes (Colab, free T4 GPU)

The writer model continues each of the 12 scene openings in `prompts.jsonl`, **256 times each**
(3,072 scenes). Back on the laptop, the gold and proxy critics score every candidate, and
best-of-N is computed exactly from these pools, for any N up to 256.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all.
You will be asked to upload `data/processed/prompts.jsonl`. At the end, `candidates.jsonl`
and `generation_meta.json` download automatically; put both in `data/generated/`.

Uses plain Hugging Face `transformers` (preinstalled on Colab). vLLM 0.34 crashed on start-up
(its kernel warm-up imports an unrelated model that needs a missing `cutlass` package), so it
is not used. Expected time: ~25–40 min. If the session disconnects, rerun: finished prompts
are kept in `candidates.jsonl` and skipped.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import json, os, time
import torch, transformers
from transformers import AutoModelForCausalLM, AutoTokenizer
from google.colab import files
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
print("torch", torch.__version__, "| transformers", transformers.__version__)

In [ ]:
if not os.path.exists("prompts.jsonl"):
    files.upload()                             # choose data/processed/prompts.jsonl
prompts = [json.loads(l) for l in open("prompts.jsonl", encoding="utf-8")]
print(len(prompts), "prompts")

In [ ]:
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # small, Apache-2.0, fits a T4 easily
M = 256                                # candidates per prompt (best-of-N up to 256)
BATCH = 128                            # sequences generated together (fits T4 memory)
TEMPERATURE = 1.0                      # plain sampling: diverse candidates
TOP_P = 0.95
MAX_NEW_TOKENS = 800                   # ~450 words of dialogue is ~700 tokens
SEED = 42

SYSTEM = "You are a staff writer on {show}, {genre}."
USER = (
    "Continue the scene below. Write the next ~450 words in the same transcript format: "
    "each spoken line as 'NAME: line', stage directions in [brackets]. Keep the same "
    "characters and tone. Output only the continuation, with no title, notes or preamble."
    "\n\nSCENE SO FAR:\n{opening}"
)

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL, padding_side="left")
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16).to("cuda")
model.eval()

def chat(p):
    msgs = [{"role": "system", "content": SYSTEM.format(**p)},
            {"role": "user", "content": USER.format(**p)}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

print(chat(prompts[0])[:600])

In [ ]:
OUT = "candidates.jsonl"
done = set()
if os.path.exists(OUT):                         # resume after a disconnect
    counts = {}
    for l in open(OUT, encoding="utf-8"):
        r = json.loads(l); counts[r["prompt_id"]] = counts.get(r["prompt_id"], 0) + 1
    done = {pid for pid, c in counts.items() if c == M}
    keep = [l for l in open(OUT, encoding="utf-8") if json.loads(l)["prompt_id"] in done]
    open(OUT, "w", encoding="utf-8").writelines(keep)
print("already done:", sorted(done))

eos_ids = {tok.eos_token_id, tok.convert_tokens_to_ids("<|im_end|>")}
t0, n_tok = time.time(), 0
for p in prompts:
    if p["prompt_id"] in done:
        continue
    enc = tok([chat(p)], return_tensors="pt").to("cuda")
    rows = []
    for b in range(M // BATCH):
        torch.manual_seed(SEED + 1000 * p["prompt_id"] + b)
        with torch.no_grad():
            gen = model.generate(**enc, do_sample=True, temperature=TEMPERATURE, top_p=TOP_P,
                                 max_new_tokens=MAX_NEW_TOKENS, num_return_sequences=BATCH,
                                 pad_token_id=tok.pad_token_id)
        new = gen[:, enc["input_ids"].shape[1]:].tolist()
        for seq in new:
            ids = [t for t in seq if t != tok.pad_token_id]
            stopped = any(t in eos_ids for t in ids)
            ids = [t for t in ids if t not in eos_ids]
            n_tok += len(ids)
            rows.append({"prompt_id": p["prompt_id"], "cand_id": len(rows),
                         "text": tok.decode(ids, skip_special_tokens=True),
                         "finish_reason": "stop" if stopped else "length", "n_tokens": len(ids)})
    with open(OUT, "a", encoding="utf-8") as f:
        f.writelines(json.dumps(r, ensure_ascii=False) + "\n" for r in rows)
    el = time.time() - t0
    print(f"prompt {p['prompt_id']:2d} done | {el / 60:.1f} min so far | {n_tok / el:,.0f} tokens/s")

meta = dict(model=MODEL, engine=f"transformers {transformers.__version__}", M=M,
            temperature=TEMPERATURE, top_p=TOP_P, max_new_tokens=MAX_NEW_TOKENS, seed=SEED,
            system=SYSTEM, user=USER, n_prompts=len(prompts),
            seconds_this_session=round(time.time() - t0, 1))
json.dump(meta, open("generation_meta.json", "w"), indent=2)

In [ ]:
import collections, statistics
rows = [json.loads(l) for l in open("candidates.jsonl", encoding="utf-8")]
assert len(rows) == M * len(prompts), f"expected {M * len(prompts)} rows, got {len(rows)}: rerun the cell above"
words = [len(r["text"].split()) for r in rows]
print(len(rows), "candidates | words per candidate: median", statistics.median(words),
      "| min", min(words), "| max", max(words))
print("finish reasons:", collections.Counter(r["finish_reason"] for r in rows))
for r in rows[:2]:
    print("\n----- prompt", r["prompt_id"], "cand", r["cand_id"], "-----\n", r["text"][:800])

In [ ]:
files.download("candidates.jsonl")
files.download("generation_meta.json")